In [ ]:
# --- Paths (repo-relative; this notebook runs from notebooks/) ---
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
DATA      = PROJECT_ROOT / 'data'
RAW       = DATA / 'raw'          # external source data
ANNOTATED = DATA / 'annotated'    # pipeline-derived tables
FIGURES   = PROJECT_ROOT / 'figures'
# NOTE: some cells below still read AKEY-cluster paths (/scratch/gpfs/... ,
# /projects/AKEY/...) for inputs not mirrored in this repo (goatools GO files,
# UniProt idmapping .parquet, HumanTFs .csv). Those cells only run on the cluster.


# APPRIS Annotation Merge (v2 — isoform-aware mapping)

Joins APPRIS module scores onto the four IDR isoform datasets.

**Mapping file:** `upkb_to_ensemble_humanisos.tsv`
- `From`: UniProt isoform accession (canonical AND alternative, e.g. `A0AUZ9`, `A0AUZ9-2`, `A0AUZ9-3`)
- `To`: Ensembl Gene ID with version (e.g. `ENSG00000144445.18`) → stripped to bare ENSG for APPRIS join

**Join chain:** `isoform_accession` → ENSG (via new mapping) → APPRIS gene-level summary

**Important limitation:** APPRIS scores are gene-level (one row per ENSG), so all UniProt
isoforms of the same gene receive the same APPRIS values. True isoform-level APPRIS scores
would require a UniProt isoform → Ensembl *transcript* (ENST) mapping, which does not exist
as a maintained cross-reference.

**Canonical transcript selection per gene (priority order):**
1. MANE_Select transcript
2. PRINCIPAL:1 (highest APPRIS score if ties)
3. Any PRINCIPAL tier (highest APPRIS score)
4. Highest APPRIS score among all translated transcripts (fallback)

**17 new columns added:**
- `ensg_id` — bare Ensembl Gene ID used for join (version stripped)
- `appris_annotation` — PRINCIPAL:1–5 | ALTERNATIVE:1–2 | MINOR (gene's canonical transcript)
- `appris_tier` — integer: 1=P1, 2=P2, 3=P3, 4=P4, 5=P5, 6=ALT1, 7=ALT2, 8=MINOR, 9=other
- `is_mane_select` — True if canonical transcript is MANE_Select
- `trifid_score` — normalized TRIFID score for canonical transcript (0–1)
- `trifid_score_max` — max TRIFID across all gene's translated transcripts
- `trifid_score_range` — trifid_score_max − trifid_score (isoform functional variation)
- `n_functional_residues` — Firestar: catalytic/ligand-binding residue count
- `structure_score` — Matador3D: structural homology coverage
- `corsair_score` — vertebrate conservation score
- `domain_score` — SPADE: Pfam domain bit-score sum
- `n_tmh` — THUMP: transmembrane helix count
- `n_proteomic_peptides` — mass-spec peptides detected
- `tsl` — transcript support level (1=best, 5=weakest)
- `protein_length_appris` — protein length in aa
- `n_transcripts_appris` — translated Ensembl transcripts for this gene
- `n_transcripts_appris_total` — all Ensembl transcripts (including non-translated)


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE       = Path('..')
ANN_DIR    = BASE / 'data' / 'annotated'
APPRIS_DIR = BASE / 'data' / 'raw' / 'appris'
MAP_FILE   = BASE / 'data' / 'raw' / 'uniprot' / 'idmapping' / 'upkb_to_ensemble_humanisos.tsv'

print('Paths configured.')

## 1. Load and Clean the UniProt Isoform → ENSG Mapping

In [ ]:
raw_map = pd.read_csv(MAP_FILE, sep='\t')
raw_map.columns = ['isoform_accession', 'ensg_versioned']

# Strip version suffix from ENSG (e.g. ENSG00000144445.18 → ENSG00000144445)
# APPRIS uses bare unversioned ENSG IDs
raw_map['ensg_id'] = raw_map['ensg_versioned'].str.split('.').str[0]

print(f'Mapping rows:                    {len(raw_map):,}')
print(f'Unique isoform accessions:       {raw_map["isoform_accession"].nunique():,}')
print(f'  With isoform suffix (-N):      {raw_map["isoform_accession"].str.contains("-").sum():,}')
print(f'  Canonical (no suffix):         {(~raw_map["isoform_accession"].str.contains("-")).sum():,}')
print(f'Unique bare ENSG IDs:            {raw_map["ensg_id"].nunique():,}')
print()

# Some isoform_accessions map to multiple ENSG IDs (e.g. gene duplications, alt loci).
# Strategy: keep only ENSG IDs that appear in APPRIS data; if still multiple, keep first.
# We resolve this after loading APPRIS below.
print('Sample rows:')
print(raw_map.head(8).to_string(index=False))

## 2. Load APPRIS and Build Gene-Level Summary

In [ ]:
ap = pd.read_csv(APPRIS_DIR / 'appris_data.appris.txt', sep='\t', low_memory=False)

ap = ap.rename(columns={
    'Ensembl Gene ID':                'ensg_id',
    'Gene name (HGNC)':               'hgnc_name',
    'Transcript ID':                  'enst_id',
    'Translation ID':                 'ensp_id',
    'Is translated?':                 'is_translated',
    'Transcript type':                'transcript_type',
    'CCDS ID':                        'ccds_id',
    'Transcript support level':       'tsl',
    'Protein length':                 'protein_length_appris',
    'Functional residues (firestar)': 'n_functional_residues',
    'Structure score (Matador)':      'structure_score',
    'Conservation (CORSAIR)':         'corsair_score',
    'Domain Score (SPADE)':           'domain_score',
    'Trans-membrane helices (THUMP)': 'n_tmh',
    'Signal sequence (CRASH)':        'signal_sequence',
    'Trifid Score':                   'trifid_score',
    'Peptides':                       'n_proteomic_peptides',
    'APPRIS score':                   'appris_score',
    'APPRIS Annotation':              'appris_annotation',
    'MANE':                           'mane_tag',
})

for col in ['tsl', 'protein_length_appris', 'n_functional_residues',
            'structure_score', 'corsair_score', 'domain_score',
            'n_tmh', 'trifid_score', 'n_proteomic_peptides', 'appris_score']:
    ap[col] = pd.to_numeric(ap[col], errors='coerce')

appris_ensg_set = set(ap['ensg_id'].unique())

print(f'APPRIS rows:         {len(ap):,}')
print(f'APPRIS unique genes: {len(appris_ensg_set):,}')
print(f'MANE_Select rows:    {(ap["mane_tag"] == "MANE_Select").sum():,}')

In [ ]:
# Resolve multi-ENSG mappings: for each isoform_accession with >1 ENSG,
# prefer the ENSG that exists in APPRIS; if still multiple, keep first occurrence.

raw_map['in_appris'] = raw_map['ensg_id'].isin(appris_ensg_set)

# For each isoform_accession: sort so in_appris=True rows come first, then keep first
iso_to_ensg = (
    raw_map
    .sort_values('in_appris', ascending=False)   # True (1) before False (0)
    .drop_duplicates(subset='isoform_accession', keep='first')
    .set_index('isoform_accession')['ensg_id']
)

multi_ensg = raw_map.groupby('isoform_accession')['ensg_id'].nunique()
print(f'Isoform accessions with >1 ENSG: {(multi_ensg > 1).sum():,}')
print(f'Final unique isoform→ENSG mappings: {len(iso_to_ensg):,}')
print(f'Of those, ENSG in APPRIS: {iso_to_ensg.isin(appris_ensg_set).sum():,}')

In [ ]:
# Build per-gene canonical APPRIS summary

def pick_canonical(group):
    mane = group[group['mane_tag'] == 'MANE_Select']
    if len(mane) > 0:
        return mane.sort_values('appris_score', ascending=False).iloc[0]
    p1 = group[group['appris_annotation'] == 'PRINCIPAL:1']
    if len(p1) > 0:
        return p1.sort_values('appris_score', ascending=False).iloc[0]
    principal = group[group['appris_annotation'].str.startswith('PRINCIPAL', na=False)]
    if len(principal) > 0:
        return principal.sort_values('appris_score', ascending=False).iloc[0]
    return group.sort_values('appris_score', ascending=False).iloc[0]

print('Building gene-level canonical summary...')
canonical_rows = ap.groupby('ensg_id', group_keys=False).apply(lambda g: pick_canonical(g.assign(ensg_id=g.name))).reset_index(drop=True)  # pandas>=3 omits the key column
canonical_rows['is_mane_select'] = canonical_rows['mane_tag'] == 'MANE_Select'

TIER_MAP = {
    'PRINCIPAL:1': 1, 'PRINCIPAL:2': 2, 'PRINCIPAL:3': 3,
    'PRINCIPAL:4': 4, 'PRINCIPAL:5': 5,
    'ALTERNATIVE:1': 6, 'ALTERNATIVE:2': 7,
    'MINOR': 8,
}
canonical_rows['appris_tier'] = canonical_rows['appris_annotation'].map(TIER_MAP).fillna(9).astype(int)

print(f'Gene-level summary rows: {len(canonical_rows):,}')
print('Tier distribution:')
tier_labels = {1:'P1',2:'P2',3:'P3',4:'P4',5:'P5',6:'ALT1',7:'ALT2',8:'MINOR',9:'OTHER'}
for tier, count in canonical_rows['appris_tier'].value_counts().sort_index().items():
    print(f'  {tier} ({tier_labels.get(tier,"?"):<6}): {count:,}')

In [ ]:
# Per-gene TRIFID stats across all translated transcripts
translated = ap[ap['is_translated'] == 'TRANSLATION']

trifid_stats = (
    translated.groupby('ensg_id')['trifid_score']
    .agg(trifid_score_max='max', n_transcripts_appris='count')
    .reset_index()
)
n_all_tx = ap.groupby('ensg_id').size().reset_index(name='n_transcripts_appris_total')

# Assemble final gene annotation table
APPRIS_COLS = [
    'ensg_id', 'appris_annotation', 'appris_tier', 'is_mane_select',
    'trifid_score', 'n_functional_residues', 'structure_score',
    'corsair_score', 'domain_score', 'n_tmh', 'n_proteomic_peptides',
    'tsl', 'protein_length_appris',
]
appris_gene = canonical_rows[APPRIS_COLS].copy()
appris_gene = appris_gene.merge(trifid_stats, on='ensg_id', how='left')
appris_gene = appris_gene.merge(n_all_tx, on='ensg_id', how='left')
appris_gene['trifid_score_range'] = appris_gene['trifid_score_max'] - appris_gene['trifid_score']

FINAL_COLS = [
    'ensg_id', 'appris_annotation', 'appris_tier', 'is_mane_select',
    'trifid_score', 'trifid_score_max', 'trifid_score_range',
    'n_functional_residues', 'structure_score', 'corsair_score', 'domain_score',
    'n_tmh', 'n_proteomic_peptides', 'tsl', 'protein_length_appris',
    'n_transcripts_appris', 'n_transcripts_appris_total',
]
appris_gene = appris_gene[FINAL_COLS]
print(f'APPRIS gene annotation table: {appris_gene.shape}')
print(appris_gene.head(3).to_string())

## 3. Load Datasets and Coverage Report

In [ ]:
DATASETS = {
    'IDRisoforms_df_geneage':                     ANN_DIR / 'IDRisoforms_df_geneage.csv',
    'IDRisoforms_df_geneage_UNFILTERED':           ANN_DIR / 'IDRisoforms_df_geneage_UNFILTERED.csv',
    'TranscriptionIsoforms_df_geneage':            ANN_DIR / 'TranscriptionIsoforms_df_geneage.csv',
    'TranscriptionIsoforms_df_geneage_UNFILTERED': ANN_DIR / 'TranscriptionIsoforms_df_geneage_UNFILTERED.csv',
}

# The *_UNFILTERED inputs have no generating code in this repository (see README); use them if present.
dfs = {name: pd.read_csv(path) for name, path in DATASETS.items() if path.exists()}
print('Loaded datasets:')
for name, df in dfs.items():
    n_iso = df['isoform_accession'].nunique()
    n_base = df['base_accession'].nunique()
    print(f'  {name}: {len(df):,} rows | {n_iso:,} unique isoforms | {n_base:,} unique genes')

In [ ]:
# ── COVERAGE REPORT ──────────────────────────────────────────────────────────
# Join key is now isoform_accession (not base_accession)
# Reports:
#   (A) isoform_accessions with no ENSG mapping
#   (B) isoform_accessions mapped to ENSG but ENSG not in APPRIS
#   (C) fully matched
#   (D) note: all isoforms of same gene get same APPRIS values

iso_map_set    = set(iso_to_ensg.index)
appris_ensg_set = set(appris_gene['ensg_id'])

print('=' * 70)
print('COVERAGE REPORT  (join key: isoform_accession)')
print('=' * 70)

for name, df in dfs.items():
    all_iso  = df['isoform_accession'].unique()
    n_unique = len(all_iso)
    n_total  = len(df)

    # (A) No ENSG mapping at all
    missing_map      = [a for a in all_iso if a not in iso_map_set]
    missing_map_rows = df['isoform_accession'].isin(missing_map).sum()

    # (B) Has ENSG but ENSG not in APPRIS
    found_iso   = [a for a in all_iso if a in iso_map_set]
    no_appris   = [a for a in found_iso if iso_to_ensg[a] not in appris_ensg_set]
    no_appris_rows = df['isoform_accession'].isin(no_appris).sum()

    # (C) Fully matched
    matched_iso  = n_unique - len(missing_map) - len(no_appris)
    matched_rows = n_total  - missing_map_rows - no_appris_rows

    # (D) How many unique genes those matched isoforms belong to
    matched_iso_set = set(found_iso) - set(no_appris)
    matched_base    = df[df['isoform_accession'].isin(matched_iso_set)]['base_accession'].nunique()

    print(f'\n── {name} ──')
    print(f'  Total rows:                     {n_total:,}')
    print(f'  Unique isoform_accessions:      {n_unique:,}')
    print(f'  (A) No ENSG mapping:            {len(missing_map):,} isoforms ({missing_map_rows:,} rows)')
    print(f'  (B) ENSG not in APPRIS:         {len(no_appris):,} isoforms ({no_appris_rows:,} rows)')
    print(f'  (C) Fully matched:              {matched_iso:,} isoforms ({matched_rows:,} rows)')
    print(f'      Covering {matched_base:,} unique genes')
    print(f'  Coverage: {matched_rows/n_total*100:.1f}% of rows')
    print(f'  NOTE: isoforms sharing a gene all receive the same APPRIS gene-level scores')

print('\n' + '=' * 70)

## 4. Merge and Save

In [ ]:
def merge_appris(df, iso_to_ensg, appris_gene):
    """
    Left-join APPRIS gene annotations onto a dataset.

    Join strategy — two-step with fallback:
      1. Try isoform_accession → ensg_id  (uses isoform-specific mapping where available)
      2. Fall back to base_accession → ensg_id for any isoform not in the mapping
         (safe because APPRIS is gene-level: all isoforms of a gene share one ENSG)
      3. Left-join appris_gene on ensg_id

    This ensures alternative isoforms whose specific accession isn't in the mapping
    still inherit their gene's APPRIS annotation via the canonical base_accession.
    """
    out = df.copy()
    # Step 1: map by isoform_accession (catches isoform-specific entries in the mapping)
    out['ensg_id'] = out['isoform_accession'].map(iso_to_ensg)
    # Step 2: fill remaining NaN using base_accession (canonical always in mapping)
    missing = out['ensg_id'].isna()
    out.loc[missing, 'ensg_id'] = out.loc[missing, 'base_accession'].map(iso_to_ensg)
    # Step 3: left-join APPRIS gene summary
    out = out.merge(appris_gene, on='ensg_id', how='left')
    n_via_iso  = (~missing).sum()
    n_via_base = (missing & out['ensg_id'].notna()).sum()
    n_still_na = out['ensg_id'].isna().sum()
    print(f'  Mapped via isoform_accession: {n_via_iso:,}')
    print(f'  Mapped via base_accession:    {n_via_base:,}')
    print(f'  No mapping found:             {n_still_na:,}')
    return out

OUT_NAMES = {
    'IDRisoforms_df_geneage':                     'IDRisoforms_df_appris',
    'IDRisoforms_df_geneage_UNFILTERED':           'IDRisoforms_df_appris_unfiltered',
    'TranscriptionIsoforms_df_geneage':            'TranscriptionIsoforms_df_appris',
    'TranscriptionIsoforms_df_geneage_UNFILTERED': 'TranscriptionIsoforms_df_appris_unfiltered',
}

merged = {}
for name, df in dfs.items():
    print(f'\n{name}:')
    merged[name] = merge_appris(df, iso_to_ensg, appris_gene)

print('\nMerge complete.')
for name, df in merged.items():
    print(f'  {name}: {df.shape}')

In [ ]:
# Null rate check on new APPRIS columns
APPRIS_NEW_COLS = [
    'ensg_id', 'appris_annotation', 'appris_tier', 'is_mane_select',
    'trifid_score', 'trifid_score_max', 'trifid_score_range',
    'n_functional_residues', 'structure_score', 'corsair_score', 'domain_score',
    'n_tmh', 'n_proteomic_peptides', 'tsl', 'protein_length_appris',
    'n_transcripts_appris', 'n_transcripts_appris_total',
]

sample = merged['IDRisoforms_df_geneage']
print('Null rates — IDRisoforms_df_geneage (filtered):')
for col in APPRIS_NEW_COLS:
    rate = sample[col].isnull().mean() * 100
    status = '✓' if rate < 5 else ('⚠' if rate < 20 else '✗')
    print(f'  {status} {col:<35} {rate:.1f}% null')

print()
sample2 = merged.get('IDRisoforms_df_geneage_UNFILTERED', sample)
print('Null rates — IDRisoforms_df_geneage_UNFILTERED:')
for col in ['appris_annotation', 'trifid_score', 'corsair_score']:
    rate = sample2[col].isnull().mean() * 100
    print(f'  {col:<35} {rate:.1f}% null')

In [ ]:
# Save all four files
for name, df in merged.items():
    out_path = ANN_DIR / f'{OUT_NAMES[name]}.csv'
    df.to_csv(out_path, index=False)
    print(f'Saved: {out_path.name}  ({len(df):,} rows, {len(df.columns)} cols)')

## 5. Missing Annotation Report

In [ ]:
# Detailed breakdown of what's still missing after the merge
for key_name in ['IDRisoforms_df_geneage', 'TranscriptionIsoforms_df_geneage']:
    df = merged[key_name]
    missing_mask = df['appris_annotation'].isna()
    missing = df[missing_mask].drop_duplicates('isoform_accession')[[
        'isoform_accession', 'base_accession', 'gene_name', 'tf_group', 'is_canonical'
    ]]

    print(f'\n── {key_name} — isoforms with NO APPRIS annotation ──')
    print(f'  Isoform rows missing:  {missing_mask.sum():,}')
    print(f'  Unique isoforms:       {len(missing):,}')
    if len(missing) > 0:
        print(f'  By tf_group:  {missing["tf_group"].value_counts().to_dict()}')
        print(f'  By canonical: {missing["is_canonical"].value_counts().to_dict()}')
        if len(missing) <= 60:
            print()
            print('  isoform_accession | base_accession  | gene_name    | tf_group | canonical')
            print('  ' + '-'*75)
            for _, r in missing.sort_values('gene_name').iterrows():
                print(f'  {r["isoform_accession"]:<20} {r["base_accession"]:<16} {r["gene_name"]:<14} '
                      f'{r["tf_group"]:<10} {r["is_canonical"]}')

In [ ]:
# Final summary
print('=' * 65)
print('FINAL SUMMARY')
print('=' * 65)
for name, df in merged.items():
    total   = len(df)
    matched = df['appris_annotation'].notna().sum()
    pct     = matched / total * 100
    print(f'\n{OUT_NAMES[name]}')
    print(f'  Total rows:        {total:,}')
    print(f'  With APPRIS:       {matched:,} ({pct:.1f}%)')
    print(f'  Without APPRIS:    {total - matched:,} ({100-pct:.1f}%)')
    if 'tf_group' in df.columns:
        for grp, gdf in df.groupby('tf_group'):
            n = gdf['appris_annotation'].notna().sum()
            print(f'    {grp:<10}: {n:,}/{len(gdf):,} ({n/len(gdf)*100:.1f}%)')
print('\nDone.')